In [1]:
import pandas as pd
from datetime import datetime

In [2]:
def fill_estimativa(row):
    estimada, verificada = row
    if estimada == 0:
        return verificada
    return estimada

In [ ]:
# SELECIONANDO 3 FOTOVOLTAICAS PARA O MVP
coff_foto = pd.read_parquet("../data/processed/coff/constrained_off_fotovoltaica_tm.parquet")
ne_geoelet = coff_foto.loc[coff_foto['id_estado'].isin(
    ["CE", "BA", "PE", "PI", "MA", "RN", "PB", "AL", "SE"])]

ne_geoelet = ne_geoelet[
    ['id_estado', 'nom_usina', 'ceg', 'din_instante',
     'val_geracao', 'val_geracaoreferencia']]

group_estado = ne_geoelet.groupby(['din_instante', 'nom_usina', 'ceg', 'id_estado']).agg(
    {'val_geracao': 'sum',
     'val_geracaoreferencia': 'sum'}).reset_index()

usinas = ['CONJ. FUTURA', 'CONJ. SOL DO SERTÃO', 'CONJ. ASSU SOL']

indv_estado = group_estado[group_estado['nom_usina'].isin(usinas)]

for usina in usinas:
    df_usina = indv_estado[indv_estado['nom_usina'] == usina]
    df_usina = df_usina.sort_values(by='din_instante').rename(columns={'din_instante': 'time'})
    df_usina.pop('val_geracao')
    df_usina.to_parquet(f"../data/processed/geracao_indv/{usina.replace(' ', '_').replace('.', '').lower()}.parquet", index=False)

In [19]:
coff_eol = pd.read_parquet("../data/processed/coff/constrained_off_eolica_tm.parquet")
ne_geoelet = coff_eol.loc[coff_eol['id_estado'].isin(
    ["CE", "BA", "PE", "PI", "MA", "RN", "PB", "AL", "SE"])]

ne_geoelet = ne_geoelet[
    ['id_estado', 'nom_usina', 'ceg', 'din_instante',
     'val_geracao', 'val_disponibilidade', 'val_geracaolimitada']]

group_estado = ne_geoelet.groupby(['din_instante', 'nom_usina', 'ceg', 'id_estado']).agg(
    {'val_geracao': 'sum',
     'val_disponibilidade': 'sum'}).reset_index()



In [25]:
df_usina = group_estado.loc[group_estado['nom_usina'].str.contains('CONJ. LAGOA DOS VENTOS')]
df_usina = df_usina.sort_values(by='din_instante').rename(columns={'din_instante': 'time'})
df_usina.pop('val_geracao')
df_usina.to_parquet(f"../data/processed/geracao_indv/{'CONJ. LAGOA DOS VENTOS'.replace(' ', '_').replace('.', '').lower()}.parquet", index=False)

In [23]:
sliced

,din_instante,nom_usina,ceg,id_estado,val_disponibilidade
70,2023-10-01 00:00:00,CONJ. LAGOA DOS VENTOS,-,PI,626.573
211,2023-10-01 00:30:00,CONJ. LAGOA DOS VENTOS,-,PI,627.578
352,2023-10-01 01:00:00,CONJ. LAGOA DOS VENTOS,-,PI,627.700
493,2023-10-01 01:30:00,CONJ. LAGOA DOS VENTOS,-,PI,627.700
634,2023-10-01 02:00:00,CONJ. LAGOA DOS VENTOS,-,PI,627.700
...,...,...,...,...,...
7300074,2026-08-31 21:30:00,CONJ. LAGOA DOS VENTOS,-,PI,710.670
7300214,2026-08-31 22:00:00,CONJ. LAGOA DOS VENTOS,-,PI,710.892
7300354,2026-08-31 22:30:00,CONJ. LAGOA DOS VENTOS,-,PI,712.429
7300494,2026-08-31 23:00:00,CONJ. LAGOA DOS VENTOS,-,PI,714.046


In [9]:
def primeiro_ultimo_registro_por_usina(
    dados,
    coluna_usina="nom_usina",
    coluna_data="din_instante",
    coluna_referencia="val_geracaoreferencia",
):
    dados = dados.assign(**{coluna_data: pd.to_datetime(dados[coluna_data])})

    return (
        dados.groupby(coluna_usina, as_index=False)
        .agg(
            primeiro_registro=(coluna_data, "min"),
            ultimo_registro=(coluna_data, "max"),
            media_val_geracao_referencia=(coluna_referencia, "mean"),
        )
        .sort_values(coluna_usina)
    )

primeiro_ultimo_registro_por_usina(group_estado).sort_values("media_val_geracao_referencia")

,nom_usina,primeiro_registro,ultimo_registro,media_val_geracao_referencia
167,CONJUNTO EÓLICO BABILONIA CENTRO,2024-12-31,2024-12-31 23:30:00,0.000000
170,CONJUNTO EÓLICO KAIROS,2024-12-20,2024-12-31 23:30:00,0.000000
3,CATAVENTOS DO ACARAÚ I,2025-01-01,2026-08-31 23:30:00,0.572273
2,CATAVENTOS ACARAÚ I,2024-01-01,2024-12-31 23:30:00,2.599904
173,EÓLICA CATAVENTOS DO ACARAÚ I,2023-10-01,2023-12-31 23:30:00,6.188283
...,...,...,...,...
64,CONJ. EÓLICO CAMPO LARGO,2024-11-06,2024-11-18 23:30:00,322.166117
87,CONJ. LAGOA DOS VENTOS,2023-10-01,2026-08-31 23:30:00,345.981887
133,CONJ. SERRA DO ASSURUÁ,2025-01-01,2026-08-31 23:30:00,357.058192
168,CONJUNTO EÓLICO CURRAL NOVO DO PIAUI II 230KV,2024-09-26,2024-12-31 23:30:00,397.614257
